# 工商管理答题助手 — LoRA 微调（Colab T4）

在来自 [github.com/ZHAO11451419/business-admin-answer-helper](https://github.com/ZHAO11451419/business-admin-answer-helper)
的 445 对工商管理答题数据集上微调 **Qwen2.5-7B-Instruct**。

- 运行环境：**T4 GPU（16GB）— 免费**（运行环境 → 更改运行时类型 → T4）
- 方法：QLoRA（4-bit）+ LoRA，T4 上约 25-40 分钟
- 产物：LoRA 适配器 + 合并后的 16-bit 模型，可选择性推送到 Hugging Face


## 1. 安装依赖
运行此单元格（约 1-2 分钟）。


In [ ]:
import subprocess, sys

# Install HF training stack + bitsandbytes for 4-bit QLoRA
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "-U", "transformers", "peft", "trl", "datasets", "accelerate",
    "bitsandbytes", "sentencepiece"])

# Check GPU
import torch
assert torch.cuda.is_available(), "GPU not detected — set Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


## 2. 加载数据集
从 GitHub 仓库下载 `train.jsonl`（445 对）并展示题型分布。


In [ ]:
import json, urllib.request
from collections import Counter

url = "https://raw.githubusercontent.com/ZHAO11451419/business-admin-answer-helper/main/data/train.jsonl"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
data = urllib.request.urlopen(req).read().decode("utf-8")

pairs = [json.loads(l) for l in data.strip().splitlines() if l.strip()]
print(f"Loaded {len(pairs)} pairs")
print(Counter(p["type"] for p in pairs))


## 3. 以 4-bit 加载基座模型（QLoRA）
以 NF4 + 双重量化将 **Qwen/Qwen2.5-7B-Instruct** 量化到 4-bit，使其能在 16GB 显存中运行。


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE = "Qwen/Qwen2.5-7B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map="auto", trust_remote_code=True)
print("Model loaded:", BASE)


## 4. 附加 LoRA 适配器
只有约 1-2% 的参数可训练。在全部 445 对数据上训练 3 个 epoch，T4 上约需 25-40 分钟。


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## 5. 训练（使用 Qwen 对话模板的监督微调）
- 有效批量 = 2 × 8 = 16
- 余弦学习率调度，warmup 10%
- 留出 5% 作为验证集
- 如需调整，可修改下方的 `EPOCHS` / `LR`。


In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

EPOCHS = 3.0
LR = 2e-4
BATCH = 2
GRAD_ACCUM = 8
MAX_LEN = 2048

def fmt(example):
    user = example["instruction"]
    if example.get("input", "").strip():
        user = user + "\n" + example["input"].strip()
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user},
         {"role": "assistant", "content": example["output"]}],
        tokenize=False, add_generation_prompt=False)

ds = Dataset.from_list(pairs)
split = ds.train_test_split(test_size=0.05, seed=42)
train_ds, val_ds = split["train"], split["test"]

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    formatting_func=fmt,
    args=SFTConfig(
        output_dir="outputs/business-admin-answer-helper",
        max_length=MAX_LEN,
        dataset_text_field="text",
        packing=False,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        lr_scheduler_type="cosine",
        warmup_steps=8,
        logging_steps=5,
        save_strategy="steps",
        save_steps=200,
        eval_strategy="steps",
        eval_steps=200,
        bf16=True,
        seed=42,
        report_to=[],
    ),
)

trainer.train()
trainer.save_model("outputs/business-admin-answer-helper/adapter")
print("Adapter saved.")


## 6. 将适配器合并进基座模型（16-bit）
合并后的模型是独立的 16-bit 模型，可以推送到 Hugging Face 并在任意环境使用。


In [ ]:
model = model.merge_and_unload()
merged_dir = "outputs/business-admin-answer-helper/merged"
model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print("Merged 16-bit model saved to", merged_dir)
print("Disk size (GB):", round(sum(os.path.getsize(os.path.join(merged_dir, f))
      for f in os.listdir(merged_dir)) / 1e9, 1))


## 7. 测试模型
用与训练数据相同的格式提问。模型应按照高分答题格式作答：明确判断、计算过程、商业解读。


In [ ]:
import os

def answer(prompt, max_new=512):
    msgs = [{"role": "user", "content": prompt}]
    p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(p, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=max_new, do_sample=False)
    return tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

q1 = "Melur Kapital Bhd is a Main Market company on Bursa Malaysia with a market capitalisation above RM2 billion. Its board asks whether it must prepare a sustainability report and how. Advise the board on the National Sustainability Reporting Framework (NSRF)."
print("Q:", q1[:80], "...")
print("A:", answer(q1))

q2 = "A company sells 10,000 units at RM20 each; variable cost is RM8 per unit and fixed costs are RM40,000. Compute the break-even point and margin of safety."
print("\nQ:", q2)
print("A:", answer(q2))


## 8.（可选）推送到 Hugging Face
1. 在 https://huggingface.co/settings/tokens 创建 token（权限选 write）
2. 运行下方单元格，按提示粘贴 token
3. 你的模型仓库：`https://huggingface.co/<用户名>/business-admin-answer-helper`

> 模型权重太大，不适合放 GitHub；GitHub 仓库保存代码 + 数据集，训练好的权重发布在 Hugging Face。


In [ ]:
# Only run if you want to publish weights
from huggingface_hub import notebook_login, HfApi
notebook_login()

REPO = "business-admin-answer-helper"
api = HfApi()
api.create_repo(repo_id=REPO, exist_ok=True)
api.upload_folder(folder_path="outputs/business-admin-answer-helper/merged",
                  repo_id=REPO, repo_type="model")
print("Pushed to https://huggingface.co/" + REPO)
